# Q4 patch teszt és hálótorzítás
Borsodi Bálint TDK-dolgozata, B1 és B7 (6.2–6.4., illetve 7.10. szakasz).
Az affin mezőt a teljes peremen előírjuk. A belső csomópontokban az egzakt megoldást kell visszakapnunk, a teljes (FI) és a szelektív redukált (SRI) integrálással is.

In [1]:
include(joinpath(@__DIR__, "q4_common.jl"))

min_jacobian (generic function with 1 method)

**B1.** Szabad, torzított elem szimmetriája és merevtestmódusai.

In [2]:
U, _ = q4_grid(1, 1; map=(x,y) -> (x + 0.2x*y, y - 0.1x*y))
for (name, K) in zip(("FI", "SRI"), q4_stiffness(U, 100e9, 0.45))
    A = Matrix(K.A)
    λ = eigvals(Symmetric(A))
    λtol = maximum(abs, λ) * 1e-12
    modes = count(x -> abs(x) < λtol, λ)
    @assert isapprox(A, A'; rtol=1e-13)
    @assert modes == 3 && minimum(λ) > -λtol
    println("$name: $modes merevtestmódus")
end

FI: 3 merevtestmódus
SRI: 3 merevtestmódus


**B7.** A dolgozat seed=1 torzításának kilenc eltolásvektora rögzítve van. A Gauss-ponti minimumot a 7.6. táblázattal vetjük össze. A Q4 leképezés Jacobi-determinánsa affin, így a sarkok ellenőrzése a teljes referenciaelemet lefedi. A már átfordult hálókon kihagyjuk a patch tesztet.

In [3]:
offsets = [
     0.0236432494   0.9009273927
    -0.7116807746   0.8972988943
    -0.3763370960  -0.1533471021
     0.6554051876  -0.1816017273
     0.0991873753  -0.9448817735
     0.5070262173   0.0762866264
    -0.3405365670   0.5768574069
    -0.3936103414  -0.0930042210
    -0.7319166055  -0.1937740271
]
ux(x,y,z) = 0.01 + 0.02x + 0.01y
uy(x,y,z) = 0.02 - 0.01x + 0.03y
table = [0.0469, 0.0339, 0.0226, 0.0128, 0.00475, 0.00197, -0.00130]
println("ampl.    min det J (Gauss)  min det J (sarok)    FI hiba   SRI hiba")
for (i, amp) in enumerate((0.0, 0.05, 0.10, 0.15, 0.20, 0.22, 0.24))
    function perturb(x, y)
        ix, iy = round(Int, x/0.5), round(Int, y/0.375)
        if 1 ≤ ix ≤ 3 && 1 ≤ iy ≤ 3
            dx, dy = offsets[ix + 3(iy-1), :]
            return x + amp*dx, y + amp*dy
        end
        return x, y
    end
    U, xy = q4_grid(4, 4; lx=2.0, ly=1.5, map=perturb)
    jg, jc = min_jacobian(xy), min_jacobian(xy; corners=true)
    @assert isapprox(jg, table[i]; rtol=0.006)
    @printf("%.2f       % .6f           % .6f", amp, jg, jc)
    if jc ≤ 0
        println("        átfordult")
        continue
    end
    bc = [BoundaryCondition(edge; field=U, ux, uy) for edge in ("left","right","top","bottom")]
    f = ∫(U ⋅ [0.0, 0.0])
    exact = vec([fun(x,y,0.0) for fun in (ux,uy), (x,y) in eachcol(xy)])
    errs = [maximum(abs, solveField(K, f; support=bc).a[:,1] - exact)
            for K in q4_stiffness(U, 100e9, 0.499)]
    @assert maximum(errs) < 1e-12
    @printf("        %.2e    %.2e\n", errs...)
end

ampl.    min det J (Gauss)  min det J (sarok)    FI hiba   SRI hiba
0.00        0.046875            0.046875        5.55e-17    5.90e-16
0.05        0.033908            0.032763        6.94e-17    1.21e-15
0.10        0.022565            0.020348        5.55e-17    9.19e-16
0.15        0.012847            0.009186        6.25e-17    4.58e-16
0.20        0.004754           -0.007326        átfordult
0.22        0.001972           -0.014483        átfordult
0.24       -0.001302           -0.021957        átfordult
